In [67]:
import cv2
import numpy as np
from concurrent.futures import ThreadPoolExecutor
import math

## to be removed before final 


In [48]:
def inspect_mask(mask, CLASS_NAMES):
    """
    Prints class IDs, class names, and pixel counts.
    """

    if mask is None:
        raise ValueError("Mask is None")

    print("Mask shape:", mask.shape)
    print("Mask data type:", mask.dtype)

    unique_values, counts = np.unique(mask, return_counts=True)

    print("\nClasses found in mask:")

    for value, count in zip(unique_values, counts):
        class_name = CLASS_NAMES.get(int(value), "unknown")
        print(f"Class ID: {int(value)} | Name: {class_name} | Pixels: {count}")


def print_mask_rows_columns(mask, start_row=0, end_row=10, start_col=0, end_col=10):
    """
    Prints selected rows and columns from the mask.

    Example:
        start_row=0, end_row=10
        start_col=0, end_col=10

    This prints a 10x10 part of the mask.
    """

    if mask is None:
        raise ValueError("Mask is None")

    print(f"\nPrinting mask rows {start_row} to {end_row - 1}")
    print(f"Printing mask columns {start_col} to {end_col - 1}")

    selected_area = mask[start_row:end_row, start_col:end_col]

    print(selected_area)

    return selected_area
def read_one_mask(mask_path):
    """
    Reads one saved mask image from path.
    This is only for testing.
    """

    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    if mask is None:
        raise FileNotFoundError(f"Mask not found at path: {mask_path}")

    return mask

In [68]:
import cv2
import numpy as np


def _clamp(value, min_value=0.0, max_value=1.0):
    return max(min_value, min(value, max_value))


def _draw_rounded_rectangle(img, pt1, pt2, color, radius=18, thickness=-1):
    """
    Draw rounded rectangle.
    """

    x1, y1 = pt1
    x2, y2 = pt2

    if thickness == -1:
        cv2.rectangle(img, (x1 + radius, y1), (x2 - radius, y2), color, -1)
        cv2.rectangle(img, (x1, y1 + radius), (x2, y2 - radius), color, -1)

        cv2.circle(img, (x1 + radius, y1 + radius), radius, color, -1)
        cv2.circle(img, (x2 - radius, y1 + radius), radius, color, -1)
        cv2.circle(img, (x1 + radius, y2 - radius), radius, color, -1)
        cv2.circle(img, (x2 - radius, y2 - radius), radius, color, -1)
    else:
        cv2.rectangle(img, (x1 + radius, y1), (x2 - radius, y2), color, thickness)
        cv2.rectangle(img, (x1, y1 + radius), (x2, y2 - radius), color, thickness)

        cv2.ellipse(img, (x1 + radius, y1 + radius), (radius, radius), 180, 0, 90, color, thickness)
        cv2.ellipse(img, (x2 - radius, y1 + radius), (radius, radius), 270, 0, 90, color, thickness)
        cv2.ellipse(img, (x1 + radius, y2 - radius), (radius, radius), 90, 0, 90, color, thickness)
        cv2.ellipse(img, (x2 - radius, y2 - radius), (radius, radius), 0, 0, 90, color, thickness)


def _draw_text_with_shadow(
    img,
    text,
    org,
    font_scale=0.7,
    color=(255, 255, 255),
    thickness=2,
    shadow_color=(0, 0, 0),
):
    x, y = org

    cv2.putText(
        img,
        text,
        (x + 2, y + 2),
        cv2.FONT_HERSHEY_SIMPLEX,
        font_scale,
        shadow_color,
        thickness + 2,
        cv2.LINE_AA,
    )

    cv2.putText(
        img,
        text,
        org,
        cv2.FONT_HERSHEY_SIMPLEX,
        font_scale,
        color,
        thickness,
        cv2.LINE_AA,
    )


def _draw_progress_bar(
    img,
    x,
    y,
    w,
    h,
    value,
    label,
    color,
    max_value=1.0,
):
    ratio = _clamp(value / max_value)

    _draw_text_with_shadow(
        img,
        label,
        (x, y - 8),
        font_scale=0.45,
        color=(220, 220, 220),
        thickness=1,
    )

    _draw_rounded_rectangle(img, (x, y), (x + w, y + h), (65, 65, 65), radius=7, thickness=-1)

    fill_w = int(w * ratio)
    if fill_w > 0:
        _draw_rounded_rectangle(img, (x, y), (x + fill_w, y + h), color, radius=7, thickness=-1)

    cv2.rectangle(img, (x, y), (x + w, y + h), (120, 120, 120), 1)

    _draw_text_with_shadow(
        img,
        f"{value:.2f}",
        (x + w + 10, y + h),
        font_scale=0.45,
        color=(255, 255, 255),
        thickness=1,
    )


def _draw_glow_circle(img, center, radius, color):
    """
    Fake glow effect using multiple transparent circles.
    """

    overlay = img.copy()

    for r, alpha in [(radius + 22, 0.12), (radius + 14, 0.18), (radius + 7, 0.25)]:
        cv2.circle(overlay, center, r, color, -1)
        img[:] = cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0)

    cv2.circle(img, center, radius, color, -1)
    cv2.circle(img, center, radius, (255, 255, 255), 2)


def draw_velocity_command(frame, command):
    """
    Premium visual overlay for robot velocity command.

    Input:
        frame: BGR image/frame
        command: dict containing:
            linear_velocity
            angular_velocity
            direction
            emergency_stop
            reason

    Output:
        frame with command dashboard overlay
    """

    output = frame.copy()
    h, w = output.shape[:2]

    linear_velocity = float(command.get("linear_velocity", 0.0))
    angular_velocity = float(command.get("angular_velocity", 0.0))
    direction = command.get("direction", "center")
    emergency_stop = bool(command.get("emergency_stop", False))
    reason = command.get("reason", "")

    # ----------------------------
    # Color palette BGR
    # ----------------------------
    bg_dark = (18, 18, 22)
    panel_dark = (30, 32, 38)
    white = (255, 255, 255)
    muted = (185, 185, 185)

    red = (0, 0, 255)
    green = (60, 220, 80)
    orange = (0, 165, 255)
    yellow = (0, 230, 255)
    cyan = (255, 255, 0)
    blue = (255, 120, 0)
    purple = (190, 80, 255)

    status_color = red if emergency_stop else green

    # ----------------------------
    # Main glass panel
    # ----------------------------
    panel_x = 24
    panel_y = 24
    panel_w = min(720, w - 48)
    panel_h = 245

    glass = output.copy()
    _draw_rounded_rectangle(
        glass,
        (panel_x, panel_y),
        (panel_x + panel_w, panel_y + panel_h),
        panel_dark,
        radius=24,
        thickness=-1,
    )
    output = cv2.addWeighted(glass, 0.78, output, 0.22, 0)

    # Border glow
    cv2.rectangle(
        output,
        (panel_x + 8, panel_y + 8),
        (panel_x + panel_w - 8, panel_y + panel_h - 8),
        status_color,
        2,
    )
    _draw_rounded_rectangle(
        output,
        (panel_x, panel_y),
        (panel_x + panel_w, panel_y + panel_h),
        status_color,
        radius=24,
        thickness=2,
    )

    # ----------------------------
    # Gradient header strip
    # ----------------------------
    header_h = 54
    header = output.copy()

    for i in range(panel_w):
        ratio = i / max(panel_w - 1, 1)
        c1 = np.array(status_color)
        c2 = np.array(purple if not emergency_stop else red)
        color = tuple((c1 * (1 - ratio) + c2 * ratio).astype(np.uint8).tolist())
        cv2.line(
            header,
            (panel_x + i, panel_y),
            (panel_x + i, panel_y + header_h),
            color,
            1,
        )

    output = cv2.addWeighted(header, 0.58, output, 0.42, 0)

    # Header text
    title = "AI VISION SAFETY CONTROLLER"
    status = "EMERGENCY STOP" if emergency_stop else "ACTIVE NAVIGATION"

    _draw_text_with_shadow(
        output,
        title,
        (panel_x + 22, panel_y + 35),
        font_scale=0.72,
        color=white,
        thickness=2,
    )

    # Status badge
    badge_w = 210
    badge_h = 34
    badge_x = panel_x + panel_w - badge_w - 20
    badge_y = panel_y + 11

    _draw_rounded_rectangle(
        output,
        (badge_x, badge_y),
        (badge_x + badge_w, badge_y + badge_h),
        bg_dark,
        radius=16,
        thickness=-1,
    )

    cv2.circle(output, (badge_x + 19, badge_y + 17), 7, status_color, -1)

    _draw_text_with_shadow(
        output,
        status,
        (badge_x + 36, badge_y + 24),
        font_scale=0.48,
        color=status_color,
        thickness=1,
    )

    # ----------------------------
    # Left command card
    # ----------------------------
    card_x = panel_x + 22
    card_y = panel_y + 75
    card_w = 285
    card_h = 135

    _draw_rounded_rectangle(
        output,
        (card_x, card_y),
        (card_x + card_w, card_y + card_h),
        (40, 42, 50),
        radius=18,
        thickness=-1,
    )

    if emergency_stop:
        action_text = "STOP"
        action_sub = "Immediate halt required"
        action_color = red
    elif direction == "left":
        action_text = "TURN LEFT"
        action_sub = "Avoid object on right"
        action_color = yellow
    elif direction == "right":
        action_text = "TURN RIGHT"
        action_sub = "Avoid object on left"
        action_color = yellow
    else:
        action_text = "GO STRAIGHT"
        action_sub = "Path is clear"
        action_color = green

    _draw_text_with_shadow(
        output,
        "COMMAND",
        (card_x + 18, card_y + 28),
        font_scale=0.45,
        color=muted,
        thickness=1,
    )

    _draw_text_with_shadow(
        output,
        action_text,
        (card_x + 18, card_y + 72),
        font_scale=0.95,
        color=action_color,
        thickness=3,
    )

    _draw_text_with_shadow(
        output,
        action_sub,
        (card_x + 18, card_y + 110),
        font_scale=0.48,
        color=white,
        thickness=1,
    )

    # ----------------------------
    # Middle velocity bars
    # ----------------------------
    meter_x = card_x + card_w + 34
    meter_y = card_y + 30
    meter_w = max(160, panel_w - 520)

    _draw_progress_bar(
        output,
        meter_x,
        meter_y,
        meter_w,
        16,
        linear_velocity,
        "Linear Velocity  m/s",
        green if not emergency_stop else red,
        max_value=1.0,
    )

    _draw_progress_bar(
        output,
        meter_x,
        meter_y + 64,
        meter_w,
        16,
        angular_velocity,
        "Angular Velocity  rad/s",
        orange,
        max_value=1.0,
    )

    # ----------------------------
    # Direction compass / mini radar
    # ----------------------------
    radar_x = panel_x + panel_w - 95
    radar_y = panel_y + 147

    _draw_glow_circle(output, (radar_x, radar_y), 42, status_color if emergency_stop else blue)

    cv2.line(output, (radar_x - 55, radar_y), (radar_x + 55, radar_y), (110, 110, 110), 1)
    cv2.line(output, (radar_x, radar_y - 55), (radar_x, radar_y + 55), (110, 110, 110), 1)

    if emergency_stop:
        cv2.line(output, (radar_x - 24, radar_y - 24), (radar_x + 24, radar_y + 24), red, 5)
        cv2.line(output, (radar_x + 24, radar_y - 24), (radar_x - 24, radar_y + 24), red, 5)
    elif direction == "left":
        cv2.arrowedLine(output, (radar_x + 28, radar_y), (radar_x - 30, radar_y), yellow, 5, tipLength=0.35)
    elif direction == "right":
        cv2.arrowedLine(output, (radar_x - 28, radar_y), (radar_x + 30, radar_y), yellow, 5, tipLength=0.35)
    else:
        cv2.arrowedLine(output, (radar_x, radar_y + 30), (radar_x, radar_y - 34), green, 5, tipLength=0.35)

    _draw_text_with_shadow(
        output,
        direction.upper(),
        (radar_x - 42, radar_y + 76),
        font_scale=0.48,
        color=white,
        thickness=1,
    )

    # ----------------------------
    # Reason footer
    # ----------------------------
    footer_x = panel_x + 22
    footer_y = panel_y + panel_h - 26
    reason_text = reason[:82]

    _draw_text_with_shadow(
        output,
        f"Reason: {reason_text}",
        (footer_x, footer_y),
        font_scale=0.48,
        color=(225, 225, 225),
        thickness=1,
    )

    # ----------------------------
    # Big navigation arrow in the driving area
    # ----------------------------
    arrow_overlay = output.copy()

    center_x = w // 2
    base_y = h - 85

    if emergency_stop:
        # Large stop marker
        cv2.circle(arrow_overlay, (center_x, base_y - 30), 68, red, -1)
        output = cv2.addWeighted(arrow_overlay, 0.35, output, 0.65, 0)

        cv2.circle(output, (center_x, base_y - 30), 68, red, 6)
        cv2.line(output, (center_x - 42, base_y - 72), (center_x + 42, base_y + 12), white, 8)

        _draw_text_with_shadow(
            output,
            "STOP",
            (center_x - 58, base_y + 75),
            font_scale=1.25,
            color=red,
            thickness=4,
        )

    else:
        if direction == "left":
            start = (center_x + 95, base_y)
            end = (center_x - 145, base_y)
            label = "STEER LEFT"
            label_x = center_x - 135
        elif direction == "right":
            start = (center_x - 95, base_y)
            end = (center_x + 145, base_y)
            label = "STEER RIGHT"
            label_x = center_x - 140
        else:
            start = (center_x, base_y + 65)
            end = (center_x, base_y - 155)
            label = "FORWARD"
            label_x = center_x - 95

        # Glow arrow
        cv2.arrowedLine(arrow_overlay, start, end, yellow, 22, tipLength=0.32)
        output = cv2.addWeighted(arrow_overlay, 0.25, output, 0.75, 0)

        cv2.arrowedLine(output, start, end, yellow, 9, tipLength=0.32)
        cv2.arrowedLine(output, start, end, white, 3, tipLength=0.32)

        _draw_text_with_shadow(
            output,
            label,
            (label_x, base_y + 90),
            font_scale=1.0,
            color=yellow,
            thickness=3,
        )

    return output

## main code 

In [111]:

def calculate_angular_velocity(
    direction,
    distance,
    max_distance,
    current_angular_velocity=0.2,
    max_angular_velocity=1.0,
):
    """
    Calculates positive angular velocity based on distance.

    If object is closer, angular velocity moves toward max_angular_velocity.
    If object is farther, angular velocity stays near current_angular_velocity.
    If direction is center, angular velocity is 0.
    """

    if direction == "center":
        return 0.0

    if distance == float("inf"):
        return 0.0

    # Keep distance between 0 and max_distance
    distance = max(0, min(distance, max_distance))

    # 1 = very close, 0 = far
    closeness = 1 - (distance / max_distance)

    angular_velocity = current_angular_velocity + closeness * (
        max_angular_velocity - current_angular_velocity
    )

    return angular_velocity
def validate_mask(mask):
    if mask is None:
        raise ValueError("Mask is None")
def get_class_pixels(mask, class_id):
    """
    Returns x and y coordinates of all pixels that belong to one class.
    """
    y_coords, x_coords = np.where(mask == class_id)    
    return x_coords, y_coords
def class_exists(x_coords):
    return len(x_coords) > 0
def calculate_pixel_distances(x_coords, y_coords, bottom_center):
    """
    Calculates distance from bottom-center point to all pixels of one class.
    """
    x_center, y_bottom = bottom_center
    distances = np.sqrt(
        (x_coords - x_center) ** 2 + (y_coords - y_bottom) ** 2
    )
    return distances
def get_nearest_pixel(x_coords, y_coords, distances):
    """
    Finds the nearest pixel and its distance.
    """
    nearest_index = np.argmin(distances)
    nearest_distance = float(distances[nearest_index])
    nearest_x = int(x_coords[nearest_index])
    nearest_y = int(y_coords[nearest_index])
    return nearest_distance, (nearest_x, nearest_y)
def get_side(nearest_x, x_center, center_tolerance=10):
    """
    Decides if nearest point is left, right, or center.
    """
    if nearest_x < x_center - center_tolerance:
        return "left"
    if nearest_x > x_center + center_tolerance:
        return "right"
    return "center"
def build_not_found_result():
    return {
        "distance": float("inf"),
        "nearest_point": None,
        "side": "not_found",
    }
def build_found_result(distance, nearest_point, side):
    return {
        "distance": distance,
        "nearest_point": nearest_point,
        "side": side,
    }
def get_bottom_center_point(frame_or_mask):
    """Returns the (x, y) coordinates of the bottom-center point of the frame or mask."""
    if frame_or_mask is None:
        raise ValueError("Frame or mask is None")
    h, w = frame_or_mask.shape[:2]
    x = w // 2
    y = h - 1
    return x, y
def process_one_class(mask, class_id, class_name, bottom_center, center_tolerance=10):
    """
    Processes only one class.
    Example: Human or Obstacle or Road.
    """
    x_coords, y_coords = get_class_pixels(mask, class_id)   #Give me all row and column positions where mask value equals class_id.
    if not class_exists(x_coords):
        return build_not_found_result()

    distances = calculate_pixel_distances(
        x_coords=x_coords,
        y_coords=y_coords,
        bottom_center=bottom_center
    )

    nearest_distance, nearest_point = get_nearest_pixel(
        x_coords=x_coords,
        y_coords=y_coords,
        distances=distances
    )
    x_center, _ = bottom_center
    nearest_x, _ = nearest_point
    side = get_side(
        nearest_x=nearest_x,
        x_center=x_center,
        center_tolerance=center_tolerance
    )

    return build_found_result(
        distance=nearest_distance,
        nearest_point=nearest_point,
        side=side
    )
def calculate_distance_and_side_from_bottom_center(
    mask,
    bottom_center,
    class_names,
    center_tolerance=10,
):
    """
    Calculates nearest distance and side for all classes.
    """

    validate_mask(mask)

    result = {}

    with ThreadPoolExecutor() as executor:
        outputs = executor.map(
            lambda item: (
                item[1],
                process_one_class(
                    mask=mask,
                    class_id=item[0],
                    class_name=item[1],
                    bottom_center=bottom_center,
                    center_tolerance=center_tolerance
                )
            ),
            class_names.items()
        )
    for class_name, output in outputs:
        result[class_name] = output
    return result
def get_class_info(distance_info, class_name):
    """
    Safely gets one class information from distance_info.
    """

    return distance_info.get(
        class_name,
        {
            "distance": float("inf"),
            "nearest_point": None,
            "side": "not_found",
        },
    )
def is_found(class_info):
    """
    Checks if class exists in the mask.
    """
    return class_info["distance"] != float("inf")
def get_opposite_direction(side):
    """
    If object is on right, turn left.
    If object is on left, turn right.
    If object is center, stay center/stop depending on rule.
    """
    if side == "right":
        return "left"

    if side == "left":
        return "right"

    return "center"

def slow_down(current_velocity, slow_factor=0.4):
    """
    Reduces current velocity.
    """
    return current_velocity * slow_factor
def stop_robot():
    """
    Returns stop command.
    """
    return 0.0

def compute_velocity_from_distances(
    CLASS_NAMES,
    distance_info,
    current_velocity,
    emergency_distance=90,
    warning_distance=180,
    speed_breaker_distance=160,
    sidewalk_distance=140,
    road_max_distance=40,
    slow_factor=0.4,
    min_linear_velocity=0.1,
    current_angular_velocity=0.0
):
    """
    Converts distance information into simple velocity command.
    """
    class_names = [
    CLASS_NAMES[1],  # Human
    CLASS_NAMES[2],  # Obstacle
    CLASS_NAMES[5],  # Speed Breaker
    CLASS_NAMES[4],  # Sidewalk
    CLASS_NAMES[3],  # Road
]
    def get_distance(class_name):
        return distance_info.get(class_name, {}).get("distance", float("inf"))

    def get_side(class_name):
        return distance_info.get(class_name, {}).get("side", "not_found")

    def opposite_direction(side):
        if side == "right":
            return "left"
        if side == "left":
            return "right"
        return "center"

    def command(linear_velocity, direction, emergency_stop, reason, distance=float("inf"), max_distance=180):
        angular_velocity = calculate_angular_velocity(
            direction=direction,
            distance=distance,
            max_distance=max_distance,
            current_angular_velocity=current_angular_velocity,  
        )

        return {
            "linear_velocity": linear_velocity,
            "angular_velocity": angular_velocity,
            "direction": direction,
            "emergency_stop": emergency_stop,
            "reason": reason,
        }

    # Get distances
    with ThreadPoolExecutor() as executor:
        distance_values = list(executor.map(get_distance, class_names))
    human_dist, obstacle_dist, speed_breaker_dist, sidewalk_dist, road_dist = distance_values
    # Get sides
    with ThreadPoolExecutor() as executor:
        side_values = list(executor.map(get_side, class_names))

    human_side, obstacle_side, speed_breaker_side, sidewalk_side, road_side = side_values

    # 1. Road missing / too far
    if road_dist > road_max_distance:
        return command(0.0, "center", True, "Road missing or too far")

    # 2. Emergency stop for human / obstacle
    if human_dist <= emergency_distance:
        return command(0.0, "center", True, "Human too close")

    if obstacle_dist <= emergency_distance:
        return command(0.0, "center", True, "Obstacle too close")

    # 3. Slow + avoid human / obstacle
    if human_dist <= warning_distance:
        return command(
            linear_velocity=max(current_velocity * slow_factor, min_linear_velocity),
            direction=opposite_direction(human_side),
            emergency_stop=False,
            reason="Human nearby - slow and avoid",
            distance=human_dist,
            max_distance=warning_distance,
        )
    if obstacle_dist <= warning_distance:
        return command(
            linear_velocity=max(current_velocity * slow_factor, min_linear_velocity),
            direction=opposite_direction(obstacle_side),
            emergency_stop=False,
            reason="Obstacle nearby - slow and avoid",
            distance=obstacle_dist,
            max_distance=warning_distance,
        )

    # 4. Sidewalk nearby
    if sidewalk_dist <= sidewalk_distance:
        return command(
            linear_velocity=max(current_velocity * slow_factor, min_linear_velocity),
            direction=opposite_direction(sidewalk_side),
            emergency_stop=False,
            reason="Sidewalk nearby - slow and steer away",
            distance=sidewalk_dist,
            max_distance=sidewalk_distance,
        )

    # 5. Speed breaker nearby
    if speed_breaker_dist <= speed_breaker_distance:
        return command(
            linear_velocity=max(current_velocity * slow_factor, min_linear_velocity),
            direction="center",
            emergency_stop=False,
            reason="Speed breaker nearby - slow down",
            distance=speed_breaker_dist,
            max_distance=speed_breaker_distance,
        )

    # 6. Safe path
    return command(
        linear_velocity=max(current_velocity, min_linear_velocity),
        direction="center",
        emergency_stop=False,
        reason="Path is safe",
        distance=float("inf"),
        max_distance=180,
    )
def make_segmented_image(image, mask, alpha=0.5):
    colors = {
        0: (0, 0, 0),        # background
        1: (255, 0, 0),      # Human
        2: (0, 255, 255),    # Obstacle
        3: (0, 165, 255),    # Road
        4: (0, 0, 255),      # Sidewalk
        5: (255, 255, 0),    # Speed Breaker
    }

    colored_mask = np.zeros_like(image)

    for class_id, color in colors.items():
        colored_mask[mask == class_id] = color

    segmented_image = cv2.addWeighted(image, 1 - alpha, colored_mask, alpha, 0)

    return segmented_image

In [76]:
def compute_speed_and_direction(
    mask,
    CLASS_NAMES,
    current_velocity=1.0,
    center_tolerance=10,
    emergency_distance=90,
    warning_distance=180,
    speed_breaker_distance=160,
    sidewalk_distance=140,
    road_max_distance=40,
    slow_factor=0.4,
    min_linear_velocity=0.1,
    current_angular_velocity=0.0,
):
    bottom_center = get_bottom_center_point(mask)
    distance_info = calculate_distance_and_side_from_bottom_center(
        mask=mask,
        bottom_center=bottom_center,
        class_names=CLASS_NAMES,
        center_tolerance=center_tolerance,
    )
    velocity_command = compute_velocity_from_distances(
        CLASS_NAMES=CLASS_NAMES,
        distance_info=distance_info,
        current_velocity=current_velocity,
        emergency_distance=emergency_distance,
        warning_distance=warning_distance,
        speed_breaker_distance=speed_breaker_distance,
        sidewalk_distance=sidewalk_distance,
        road_max_distance=road_max_distance,
        slow_factor=slow_factor,
        min_linear_velocity=min_linear_velocity,
        current_angular_velocity=current_angular_velocity,
    )

    return velocity_command



In [72]:
import cv2
import sys
sys.path.append("/Users/ahmadgill/Documents/GitHub/Real-Time-Semantic-Segmentation-and-Vision-based-Heading-Controller")
from mask_from_frame import mask_from_frame
CLASS_NAMES = {
    0: "background",
    1: "Human",
    2: "Obstacle",
    3: "Road",
    4: "Sidewalk",
    5: "Speed Breaker",
}
frame = cv2.imread("/Users/ahmadgill/Documents/GitHub/Real-Time-Semantic-Segmentation-and-Vision-based-Heading-Controller/input_vid_frame0.png")
mask = mask_from_frame(frame)
command = compute_speed_and_direction(
    mask=mask,
    CLASS_NAMES=CLASS_NAMES,
    current_velocity=1.0,
    center_tolerance=10,
    emergency_distance=90,
    warning_distance=180,
    speed_breaker_distance=160,
    sidewalk_distance=140,
    road_max_distance=40,
    slow_factor=0.4,
)

print(command)



{'linear_velocity': 0.0, 'angular_velocity': 0.0, 'direction': 'center', 'emergency_stop': True, 'reason': 'Human too close'}


In [114]:
import cv2
import sys

sys.path.append("/Users/ahmadgill/Documents/GitHub/Real-Time-Semantic-Segmentation-and-Vision-based-Heading-Controller")

from mask_from_frame import mask_from_frame


CLASS_NAMES = {
    0: "background",
    1: "Human",
    2: "Obstacle",
    3: "Road",
    4: "Sidewalk",
    5: "Speed Breaker",
}


def process_video_to_segmented_video(
    input_video_path,
    output_video_path,
    current_velocity=1.0,
):
    """
    Reads input .mp4 video frame by frame,
    generates segmentation mask,
    computes velocity command,
    draws segmented overlay + command,
    and saves output .mp4 video.
    """

    cap = cv2.VideoCapture(input_video_path)

    if not cap.isOpened():
        raise ValueError(f"Could not open video: {input_video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")

    out = cv2.VideoWriter(
        output_video_path,
        fourcc,
        fps,
        (width, height),
    )

    frame_index = 0

    while True:
        ret, frame = cap.read()

        if not ret:
            break
        mask = mask_from_frame(frame)

        # 2. Compute speed + direction command
        command = compute_speed_and_direction(
            mask=mask,
            CLASS_NAMES=CLASS_NAMES,
            current_velocity=current_velocity,
            center_tolerance=10,
            emergency_distance=90,
            warning_distance=180,
            speed_breaker_distance=160,
            sidewalk_distance=140,
            road_max_distance=40,
            slow_factor=0.4,
            min_linear_velocity=0.2,
            current_angular_velocity=0.0,
        )

        # 3. Make segmented image
        segmented = make_segmented_image(frame, mask)

        # 4. Draw command on segmented frame
        frame_with_command = draw_velocity_command(segmented, command)

        # 5. Write frame to output video
        out.write(frame_with_command)

        frame_index += 1

        print(f"Processed frame: {frame_index}", end="\r")

    cap.release()
    out.release()

    print(f"\nDone. Saved segmented video: {output_video_path}")

In [115]:
input_video = "/Users/ahmadgill/Documents/GitHub/Real-Time-Semantic-Segmentation-and-Vision-based-Heading-Controller/input_vid.mp4"

output_video = "/Users/ahmadgill/Documents/GitHub/Real-Time-Semantic-Segmentation-and-Vision-based-Heading-Controller/segmented_output.mp4"

process_video_to_segmented_video(
    input_video_path=input_video,
    output_video_path=output_video,
    current_velocity=1.0,
)

Processed frame: 1607
Done. Saved segmented video: /Users/ahmadgill/Documents/GitHub/Real-Time-Semantic-Segmentation-and-Vision-based-Heading-Controller/segmented_output.mp4
